# Bond_Sim: the doom-loop thesis, one notebook per phase

**Question.** How likely is a US sovereign-debt doom loop (debt growth pushes yields up, higher yields push interest
cost and future debt growth up), what interrupts it, who is hurt if nothing does, and for how long.

**How to work here.** Each phase notebook has a parameters cell at the top (`AS_OF`, `K`, `FORCE_REFRESH`). Change it and
rerun; every cell below recomputes from the point-in-time data. Expensive steps are memoized in `runs/notebook_cache/`
(delete it or set `FORCE_REFRESH = True` to rebuild). The code the cells call lives in `src/bond_sim/` where the tests
can reach it; the mathematics is in `docs/math.md`; every design decision and the evidence for it is in `docs/decisions/`.

| Phase | Notebook | What it establishes |
|---|---|---|
| 1 | `01_data_and_vintages` | what was known when: full ALFRED vintages, the Treasury ledger, the facts that had to be probed |
| 2 | `02_bond_book` | every Treasury tranche on one monthly grid, reconciled to what the Treasury publishes; the bottom-up effective rate |
| 3 | `03_dependence_factors_states` | nothing is independent: correlations, regimes, block factors, the latent behavioral chain |
| 4 | `04_sustainability` | the identity, not a threshold: pb*, g*, the decomposition of first/second/third differences, feasible pb, the trigger on history |
| 5 | `05_simulation` | two macro blocks on identical debt accounting; distributions, admissibility, trigger probabilities |
| 6 | `06_policies_and_shocks` | five policies x four shocks on common random numbers; act now vs act later |
| 7 | `07_who_gets_hurt_how_long` | sector employment paths and recovery half-lives |

**Standing rules.** No look-ahead (every loader takes `as_of`, the invariance test proves it). No independent parameters
(shock structure is estimated). Paths are filtered by feasibility, never clipped. Every policy and shock runs on the same
random numbers. Every artifact carries the configuration hash.

In [1]:
from pathlib import Path
from IPython.display import Markdown, display
display(Markdown(Path("../PARAMETERS.md").read_text()))

# Parameters that need a human decision

Everything below is a judgment call, not something the data settles. Each has
a working placeholder so the pipeline runs end to end today; none of the
placeholders should appear in the paper without being replaced or defended.
Code locations are given so a decision is a one-line change.

| ID | Parameter | Placeholder | Where | Why it matters |
|---|---|---|---|---|
| P-01 | Risk-premium slope (bps of 10y yield per 1 pt of debt/GDP above the anchor) | 2.0 | `config/default.yaml` `doomloop.risk_premium.bps_per_pct_debt_gdp` | The single number the doom-loop probability is most sensitive to. Literature range is wide (roughly 2-5 bps per pt in most estimates for advanced economies; higher for emerging markets). The paper should sweep it, not assert it. |
| P-02 | Threshold-model kink and extra slope | 130% debt/GDP, +6 bps/pt | `doomloop.risk_premium.threshold_*` | Only used if `model: threshold`. Encodes the nonlinearity claim (Reinhart-Rogoff style). Needs a cited basis or should stay as a sensitivity case. |
| P-03 | Primary-deficit anchor path | VAR mean (historical average pb) | `sim/macro.py` (VAR intercept) | The VAR mean-reverts pb to its historical average. A CBO-baseline anchor (deficits ~3% of GDP ex-interest) is the alternative and changes the level of every debt path. |
| P-04 | Doom-loop trigger (replaced 2026-09-15, Tanishk's design) | identity-based: debt-stabilizing pb* exceeds the feasible primary balance AND r > g (growth as a trailing 12-month mean), for 12 consecutive months; feasible = 90th percentile of the historical primary balance; fiscal-reaction-with-fatigue as sensitivity | `doomloop.trigger_persistence_months`, `feasible_benchmark`, `feasible_quantile`, `trigger_require_r_gt_g`, `trigger_growth_smoothing_months`; `sim/sustainability.py` | No debt level is chosen by hand. On history, quantile 1.0 (the 2000 primary surplus of 4.4% of GDP, at a 40% debt ratio) flags one quarter since 1982 and the reaction function flags all of them (it only says the ratio is rising); requiring r > g isolates the explosive case. With raw quarterly growth the probability swung from 0.99 to 0.01 between 6- and 24-month persistence because every one-quarter dip read as r > g; smoothing growth over the effective rate's own window fixes the comparison. Persistence, quantile, smoothing, and the r > g requirement are the free choices; notebooks 04 and 05 sweep them. The old 150% / 30% thresholds are kept only as a comparison column. |
| P-05 | Sector employment betas: estimated vs literature | estimated from history (`labor.beta_source`) | `sim/labor.py` | Estimated betas are honest but noisy for small sectors; literature elasticities are cleaner but imported. Report both. |
| P-06 | No-layoff mandate mechanics | floor at 0 pt/qtr rise in u during stress; growth penalty 0.5 pct pts per pt of prevented unemployment; stress = premium > 0 and u rising over 4 quarters; 24-month window | `sim/policy.py` `NoLayoffMandate`, `policy.no_layoff_*` | This is the policy the thesis was motivated by. Every number here is invented; the mechanism choice (floor vs propensity dampener) is a modeling stance. |
| P-07 | Austerity target and fiscal multiplier | primary surplus target 2% of GDP over a 12-quarter ramp; multiplier 1.0 | `policy.austerity_primary_balance_pct_gdp`, `Austerity.multiplier` | Multiplier size decides whether austerity helps or deepens the loop (the Greek debate). |
| P-08 | Monetization share and inflation cost | Fed absorbs 50% of net issuance; +2 pct pts nominal growth | `policy.monetization_share`, `Monetization.inflation_uplift_pct` | The inflation cost is what makes this a tradeoff rather than a free lunch; it is not derived from anything yet. |
| P-09 | Growth-led uplift | +0.5 pct pts trend growth | `policy.growth_uplift_pct` | Encodes an optimistic scenario; should be tied to a productivity/immigration argument. |
| P-10 | Recovery episodes | Volcker 1981, early 1990s, dot-com 2001, GFC 2008, Covid 2020 | `sim/recovery.py` `EPISODES` | Which historical recoveries are comparable to a fiscal-crisis downturn. Greece needs a non-FRED source. |
| P-11 | Refinancing tenor of the debt stock | estimated: face-weighted average remaining maturity of the outstanding stock at as_of (`sim/setup.py`) | `InitialState.avg_new_maturity_months` (override) | Sets how fast the stock reprices to new yields. Gross-issuance-weighted term (~15 months, bills dominate) was rejected as the wrong object; the stock's remaining maturity is what governs repricing. |
| P-12 | Regime-switching covariance | off | `sim/macro.py` (single covariance) | The 2026-09-15 analysis found pair-specific regimes (mean adjusted Rand 0.11 across the twelve widest pairs), so a single covariance is the evidence-backed default for now. |
| P-13 | Long-run 10y anchor | VAR sample mean 1985-2026 (reported at run time) | `doomloop.r10_anchor_pct` | The levels VAR mean-reverts the base 10y to this. A view that r* has shifted belongs here, explicitly. |
| P-14 | Natural rate of unemployment | VAR sample mean 1985-2026 | `doomloop.u_anchor_pct` | Same logic for unemployment; CBO's NAIRU estimate is the natural candidate. |
| P-15 | Inflation as a state variable | not in the VAR; present in the state model (prices block) | `sim/macro.py` `VARS`; `analysis/factors.py` | The VAR is estimated on *nominal* growth with no inflation variable, so the monetization policy's inflation uplift behaves like real growth (unemployment falls to its floor, first real run). The state block reconstructs inflation from the prices factor; the VAR block still needs a sixth variable for a credible monetization scenario. |
| P-16 | Premium to financial conditions | 1 pt of fiscal premium = 1 pt of Baa spread widening, mapped through the Baa loading on the financial factor (signed: about -1.2 factor sd per point, negative = tighter) | `states.premium_to_financial`; `sim/macroblock.py` `StateBlock` | The state model's doom-loop channel: how much a fiscal premium tightens conditions. With the hazard off (P-19) it reaches activity only through the estimated cross-block dynamics. Estimable from episodes where sovereign spreads moved (2011 downgrade, 2023 Fitch) if a term-premium proxy is used. |
| P-17 | Latent-state count and transition prior | BIC over 2-4 states subject to a minimum occupancy; sticky Dirichlet prior strength 2 | `states.n_states`, `states.candidates`, `states.prior_strength` | With ~146 quarters, the chain's transition matrix is sparse; the prior keeps regimes persistent. Report results at 2 and 3 states. |
| P-18 | Outlier treatment for the chain | factors winsorized at +/- 3.5 sd for estimation; a state count is admissible only if every state has >= 8 quarters | `states.winsor_sd`, `states.min_state_occupancy` | Without this, BIC chose four states with 2020Q2 alone in state 0 (factor means -8 to -9): a "recession state" calibrated on one quarter. The winsorized COVID residual stays in the bootstrap bank, so a COVID-scale shock can still occur; it just does not define a regime. |
| P-19 | Stress-hazard slope on financial conditions | estimated with a >= 0 sign restriction; on the 1990-2026 sample the unrestricted estimate is negative (the three stress entries, 2001, 2007Q4, 2020, were preceded by loose conditions), so the channel is off by default | `states.hazard_beta` | Three events cannot identify a hazard. With the channel off, the fiscal premium still reaches activity through the estimated cross-block VAR(1) dynamics; notebook 05's grid shows what an imposed slope adds. A value here is a stated view, e.g. from the sovereign-stress literature, not an estimate. |

## How to record a decision

1. Change the value in `config/default.yaml` (or the named code location).
2. Add a line to `docs/decisions/` explaining the source (paper, expert, own
   estimate) and the sensitivity range the paper will report.
3. Rerun `bond_sim simulate`; every artifact carries the config hash so the
   old and new runs stay distinguishable.


In [2]:
import bond_sim.config as c
cfg = c.load(); print("config hash:", cfg.content_hash())
for name in ("data", "calendar", "book", "correlation", "sim", "doomloop", "states", "labor", "policy", "recovery"):
    print(f"{name:12s}", getattr(cfg, name))

config hash: d9222bd624fd
data         DataConfig(history_start='1980-01-01', as_of=None, vintages=True, fred_key_env='FRED_API_KEY', cache=True)
calendar     CalendarConfig(freq='M', horizon_years=30)
book         BookConfig(include_tips=True, include_frn=True, include_cmb=True, coupon_months=6, face_units=1000000.0, validate_against_mspd=True)
correlation  CorrelationConfig(rolling_window_months=60, max_lag_months=24, regime_method='quantile', n_regimes=3, fdr_alpha=0.05, transform='diff')
sim          SimConfig(n_paths=5000, seed=42, dt_months=1, max_rate_pct=25.0)
doomloop     DoomLoopConfig(var_start='1985-01-01', var_max_lag=4, pb_anchor_pct_gdp=None, r10_anchor_pct=None, u_anchor_pct=None, risk_premium=RiskPremiumConfig(model='linear', bps_per_pct_debt_gdp=2.0, anchor_debt_gdp_pct=100.0, threshold_debt_gdp_pct=130.0, threshold_extra_bps_per_pct=6.0), trigger_persistence_months=12, feasible_benchmark='envelope', feasible_quantile=0.9, trigger_require_r_gt_g=True, trigger_growth_s